# Test Exhaustif des 30 MVPs

Ce notebook exécute le moteur de calcul sur la liste des 30 affirmations. Les IDBank manquants devront être ajoutés manuellement pour finaliser les 30 tests.

In [ ]:
import sys
from pathlib import Path

# Ajouter le répertoire racine au PYTHONPATH
root_dir = Path().resolve().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.providers.insee_bdm import InseeBdmProvider
from src.engine.calculator import Calculator
import pandas as pd

provider = InseeBdmProvider(use_cache=True)
print("Moteur initialisé avec cache.")

In [ ]:
# Base de données des 30 MVPs
# On renseigne les idbank connus (ex: inflation générale, chômage)
mvps = [
    {"id": "MVP-001", "desc": "Inflation générale à 12 %", "idbank": None},
    {"id": "MVP-002", "desc": "Inflation générale de 5,8 % en août 2022", "idbank": "001763852", "op": "relative", "start": "2021-08", "end": "2022-08", "expected": 5.8},
    {"id": "MVP-003", "desc": "Inflation alimentaire de 7,7 % en août 2022", "idbank": None},
    {"id": "MVP-004", "desc": "Prix de l’énergie en hausse de 22,2 %", "idbank": None},
    {"id": "MVP-005", "desc": "Ralentissement de l’inflation entre juillet et août 2022", "idbank": "001763852", "op": "relative", "start": "2021-07", "end": "2022-08"},
    {"id": "MVP-006", "desc": "Inflation moyenne de 5,2 % en 2022", "idbank": None},
    {"id": "MVP-007", "desc": "Inflation de 5,7 % en mars 2023", "idbank": "001763852", "op": "relative", "start": "2022-03", "end": "2023-03", "expected": 5.7},
    {"id": "MVP-008", "desc": "France moins inflationniste que la zone euro", "idbank": None},
    {"id": "MVP-009", "desc": "Chômage passé de 9,3 % à 7,4 %", "idbank": "001688527", "op": "point", "start": "2017-Q1", "end": "2021-Q4", "expected": -1.9},
    {"id": "MVP-010", "desc": "Plus bas niveau de chômage depuis quinze ans", "idbank": "001688527", "op": "value", "start": "2021-Q4", "expected": 7.4},
    {"id": "MVP-011", "desc": "Chômage des jeunes au plus bas depuis quarante ans", "idbank": "001688535", "op": "value", "start": "2021-Q4"},
    {"id": "MVP-012", "desc": "Plus de six millions de chômeurs", "idbank": None},
    {"id": "MVP-013", "desc": "Aucun recul du chômage", "idbank": None},
    {"id": "MVP-014", "desc": "Un million d’emplois nets créés entre 2017 et 2022", "idbank": None},
    {"id": "MVP-015", "desc": "Créations d’emplois du jamais vu", "idbank": None},
    {"id": "MVP-016", "desc": "Taux d’activité à son plus haut niveau", "idbank": None},
    {"id": "MVP-017", "desc": "723 000 naissances en 2022", "idbank": None},
    {"id": "MVP-018", "desc": "Plus faible nombre de naissances depuis 1946", "idbank": None},
    {"id": "MVP-019", "desc": "Baisse de 19 000 naissances en un an", "idbank": None},
    {"id": "MVP-020", "desc": "Fécondité de 1,80 enfant par femme en 2022", "idbank": None},
    {"id": "MVP-021", "desc": "68 millions d’habitants au 1er janvier 2022", "idbank": None},
    {"id": "MVP-022", "desc": "Population maximale de 69,3 millions en 2044", "idbank": None},
    {"id": "MVP-023", "desc": "68,1 millions d’habitants en 2070", "idbank": None},
    {"id": "MVP-024", "desc": "Fécondité de 2,3 contre 1,7 selon le lieu de naissance", "idbank": None},
    {"id": "MVP-025", "desc": "Plus d’un million d’entreprises créées en 2023", "idbank": None},
    {"id": "MVP-026", "desc": "Recul annuel de 1 % en 2023", "idbank": None},
    {"id": "MVP-027", "desc": "Recul mensuel de 1,6 % en décembre 2023", "idbank": None},
    {"id": "MVP-028", "desc": "Recul de 4,1 % des entreprises classiques", "idbank": None},
    {"id": "MVP-029", "desc": "Stabilité des immatriculations de micro-entrepreneurs", "idbank": None},
    {"id": "MVP-030", "desc": "Rupture méthodologique des créations depuis 2022", "idbank": None},
]

results = []

for m in mvps:
    if m.get("idbank") is None:
        results.append({"id": m["id"], "statut": "À COMPLÉTER", "resultat": "idbank manquant"})
        continue
        
    try:
        obs = provider.fetch_series_by_idbank(m["idbank"])
        op = m.get("op")
        val = None
        
        if op == "relative":
            val = Calculator.relative_variation(obs, m["start"], m["end"])
        elif op == "point":
            val = Calculator.point_variation(obs, m["start"], m["end"])
        elif op == "value":
            val = Calculator.get_value(obs, m["start"])
            
        expected = m.get("expected", "?")
        statut = "OK" if (val is not None) else "ERREUR"
        
        results.append({"id": m["id"], "statut": statut, "resultat": f"{val} (attendu: {expected})"})
    except Exception as e:
        results.append({"id": m["id"], "statut": "ERREUR API", "resultat": str(e)})

pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', 50)
df_results = pd.DataFrame(results)
display(df_results)